# Generating Synthetic Reviews with Sentiment

A common real-world use of a foundation model: **generating labeled test data**. Here we ask Amazon Nova Lite to write city reviews with a *known* sentiment (positive / negative / neutral) and star rating, then store them in DynamoDB. The result is a dataset you could use to test a sentiment classifier - you already know the "right answer" for each review.

This notebook walks through the four pieces:
1. A sentiment distribution across cities
2. Sentiment-specific prompts
3. The `call_nova` helper (InvokeModel with Nova's payload)
4. The main loop that ties it together and stores results

At the end there's a **cleanup** cell to delete the DynamoDB table.

In [ ]:
import json
import random
import uuid
from decimal import Decimal

import boto3

CITIES = ["Denver", "Seattle", "Austin"]
MODEL_ID = "amazon.nova-lite-v1:0"
TABLE_NAME = "synthetic-reviews"
REGION = "us-east-1"

bedrock_client = boto3.client("bedrock-runtime", region_name=REGION)
dynamodb = boto3.resource("dynamodb", region_name=REGION)

## Slide 1: Generating sentiment test data (the plan)

The overall flow is a loop over cities: pick a sentiment, build a prompt, call the model, store the result. We'll build each piece, then run the loop at the end.

> **Teaching/Learning Tip:** This is the value of synthetic data - we control the sentiment *before* generating, so every review comes pre-labeled. No manual tagging required.

First, assign a sentiment to each city. We weight it toward positive because real-world reviews skew positive - good practice for realistic test data.

In [ ]:
def generate_sentiment_distribution():
    sentiments = ["positive", "negative", "neutral"]
    weights = [0.6, 0.2, 0.2]
    return random.choices(sentiments, weights=weights, k=len(CITIES))


generate_sentiment_distribution()

## Slide 2: Prompts for sentiment test data

The sentiment is baked into the **prompt**. Each branch also returns a matching star rating, so the label stays consistent with the text.

> **Teaching/Learning Tip:** This is prompt engineering for controlled output. We keep two prompt variations per sentiment and pick one at random, so the reviews don't all sound identical. `temperature` (set later) adds further variety.

In [ ]:
def generate_review_prompt(city, sentiment):
    if sentiment == "positive":
        prompts = [
            f"Write an enthusiastic 4- or 5-star review of {city}. Focus on the "
            f"food scene, culture, and attractions. Be specific. About 150 words.",
            f"Write a glowing review of {city} - great neighborhoods, friendly "
            f"people, excellent dining. Personal and positive. About 150 words.",
        ]
        rating = random.choice([4, 4, 5])  # more 4s than 5s
    elif sentiment == "negative":
        prompts = [
            f"Write a disappointed 1- or 2-star review of {city}. Focus on cost, "
            f"traffic, weather, or crowds. Specific and honest. About 150 words.",
            f"Write a critical review of {city} - overrated attractions, poor "
            f"value, daily annoyances. About 150 words.",
        ]
        rating = random.choice([1, 2, 2])
    else:  # neutral
        prompts = [
            f"Write a balanced 3-star review of {city} weighing good and bad "
            f"evenly. Some highlights, some drawbacks. About 150 words.",
            f"Write an even-handed review of {city} - ups and downs, neither rave "
            f"nor rant. About 150 words.",
        ]
        rating = 3

    return random.choice(prompts), rating


prompt, rating = generate_review_prompt("Denver", "positive")
print("rating:", rating)
print(prompt)

## Slides 3 & 4: Invoke the model (`call_nova`)

This calls Nova Lite with `invoke_model`. Note Nova's **model-specific payload**: `schemaVersion: "messages-v1"`, a `messages` list, and `inferenceConfig`. The generated text comes back at `output.message.content[0].text`.

> **Teaching/Learning Tip:** Compare this to the Claude `invoke_model` demo - the payload shape is different for each model family. This is exactly why the Converse API exists: it gives one shape that works across models. `invoke_model` is the lower-level, model-specific path.

In [ ]:
def call_nova(prompt, bedrock_client, model_id=MODEL_ID):
    # Define the message with proper Nova Lite format
    message_list = [{"role": "user", "content": [{"text": prompt}]}]

    # Configure inference parameters
    inf_params = {"maxTokens": 400, "temperature": 0.7}

    body = {
        "schemaVersion": "messages-v1",
        "messages": message_list,
        "inferenceConfig": inf_params,
    }

    # Invoke Amazon Nova Lite with Amazon Bedrock
    response = bedrock_client.invoke_model(modelId=model_id, body=json.dumps(body))

    # Extract response body and return the content
    response_body = json.loads(response["body"].read())
    return response_body["output"]["message"]["content"][0]["text"]


print(call_nova("Write a 20-word review of Denver.", bedrock_client))

## Store results in DynamoDB

We create a table (on-demand billing, so no capacity planning) and write one row per review with its known sentiment and rating.

> **Teaching/Learning Tip:** We use the boto3 **resource** interface for DynamoDB - it converts Python types automatically. Numbers must be `Decimal` (DynamoDB stores numbers as exact decimals, not floats).

In [ ]:
def get_or_create_table():
    existing = [t.name for t in dynamodb.tables.all()]
    if TABLE_NAME in existing:
        return dynamodb.Table(TABLE_NAME)
    print(f"Creating table '{TABLE_NAME}' ...")
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "review_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "review_id", "AttributeType": "S"}],
        BillingMode="PAY_PER_REQUEST",
    )
    table.wait_until_exists()
    return table


table = get_or_create_table()
print("Table ready:", table.name)

## The main loop

Now everything comes together: distribute sentiments, then for each city build a prompt, call Nova, and store the review.

In [ ]:
sentiments = generate_sentiment_distribution()

for i, city in enumerate(CITIES):
    print(f"\nProcessing {i + 1}: {city} ({sentiments[i]})")
    prompt, rating = generate_review_prompt(city, sentiments[i])
    review_text = call_nova(prompt, bedrock_client)

    review_id = str(uuid.uuid4())
    table.put_item(Item={
        "review_id": review_id,
        "city": city,
        "sentiment": sentiments[i],
        "rating": Decimal(rating),
        "review_text": review_text,
    })
    print(f"Stored {review_id} ({rating} stars)")
    print(review_text)

Read the rows back to confirm they landed in the table:

In [ ]:
for item in table.scan()["Items"]:
    print(item["city"], "|", item["sentiment"], "|", int(item["rating"]), "stars")

## Cleanup

Delete the table when you're done so it doesn't linger in your account. Run this cell at the end of the demo.

> **Teaching/Learning Tip:** Always clean up demo resources. On-demand DynamoDB costs almost nothing at rest, but leaving orphaned tables around is a bad habit - and a real bill in a busy account.

In [ ]:
table.delete()
table.wait_until_not_exists()
print(f"Deleted table '{TABLE_NAME}'.")